# Notebook 1: Terraform Provisioning (Pre-`node1`)

Run this notebook **before** `node1` is accessible. It provisions Chameleon infrastructure from `devops/tf/kvm` and prepares the `node1` entrypoint used by the cluster bootstrap notebook.

Manual supervision points:
- Validate Terraform plan output before apply.
- Confirm Terraform outputs contain the expected `node1` access details.
- Confirm `gpu_private_ips` includes `gpu-node1` and `gpu-node2` when GPU creation is enabled.


## Chameleon Credentials Requirement

Before provisioning, obtain Chameleon Cloud application credentials and place them in:

- `~/.config/openstack/clouds.yaml`

Terraform provisioning in this notebook **cannot proceed** without that file.

In [ ]:
set -euo pipefail

REPO_ROOT="${REPO_ROOT:-$HOME/recipe-scraper-mlops}"
TF_DIR="${TF_DIR:-$REPO_ROOT/devops/tf/kvm}"
ANSIBLE_DIR="${ANSIBLE_DIR:-$REPO_ROOT/devops/ansible}"
KUBESPRAY_DIR="${KUBESPRAY_DIR:-$ANSIBLE_DIR/k8s/kubespray}"
KUBESPRAY_INVENTORY="${KUBESPRAY_INVENTORY:-$ANSIBLE_DIR/k8s/inventory/mycluster/hosts.yaml}"

TF_OPENSTACK_CLOUD="${TF_OPENSTACK_CLOUD:-kvm_tacc}"
TF_OPENSTACK_REGION="${TF_OPENSTACK_REGION:-KVM@TACC}"
TF_OPENSTACK_ENDPOINT_TYPE="${TF_OPENSTACK_ENDPOINT_TYPE:-public}"
TF_SUFFIX="${TF_SUFFIX:-proj22}"
TF_CPU_FLAVOR_ID="${TF_CPU_FLAVOR_ID:-7df7c35e-b47d-4164-a9b7-148ba76885f3}"
# Fallback single flavor used for every GPU node when per-node map is not set.
TF_GPU_FLAVOR_ID="${TF_GPU_FLAVOR_ID:-}"
# Per-node GPU flavor map JSON. Example: {"gpu-node1":"<flavor-a>","gpu-node2":"<flavor-b>"}
TF_GPU_FLAVOR_IDS_JSON="${TF_GPU_FLAVOR_IDS_JSON:-{}}"
TF_CREATE_GPU_NODE="${TF_CREATE_GPU_NODE:-true}"

cat <<'EOF'
Parameter summary:
  REPO_ROOT              = $REPO_ROOT
  TF_DIR                 = $TF_DIR
  ANSIBLE_DIR            = $ANSIBLE_DIR
  KUBESPRAY_DIR          = $KUBESPRAY_DIR
  KUBESPRAY_INVENTORY    = $KUBESPRAY_INVENTORY
  TF_OPENSTACK_CLOUD     = $TF_OPENSTACK_CLOUD
  TF_OPENSTACK_REGION    = $TF_OPENSTACK_REGION
  TF_ENDPOINT_TYPE       = $TF_OPENSTACK_ENDPOINT_TYPE
  TF_SUFFIX              = $TF_SUFFIX
  TF_CPU_FLAVOR_ID       = $TF_CPU_FLAVOR_ID
  TF_GPU_FLAVOR_ID       = $TF_GPU_FLAVOR_ID
  TF_GPU_FLAVOR_IDS_JSON = $TF_GPU_FLAVOR_IDS_JSON
  TF_CREATE_GPU_NODE     = $TF_CREATE_GPU_NODE
EOF


In [ ]:
set -euo pipefail

for cmd in terraform bash python; do
  command -v "$cmd" >/dev/null || {
    echo "Missing required command: $cmd" >&2
    exit 1
  }
done

CLOUDS_YAML="${HOME}/.config/openstack/clouds.yaml"
if [[ ! -f "$CLOUDS_YAML" ]]; then
  echo "Missing credentials file: $CLOUDS_YAML" >&2
  exit 1
fi

for path in \
  "$TF_DIR/main.tf" \
  "$TF_DIR/variables.tf" \
  "$TF_DIR/outputs.tf" \
  "$TF_DIR/provider.tf" \
  "$TF_DIR/data.tf"; do
  [[ -f "$path" ]] || {
    echo "Missing expected Terraform file: $path" >&2
    exit 1
  }
done

if [[ "$TF_CREATE_GPU_NODE" != "true" && "$TF_CREATE_GPU_NODE" != "false" ]]; then
  echo "TF_CREATE_GPU_NODE must be true or false. Current value: $TF_CREATE_GPU_NODE" >&2
  exit 1
fi

python - <<'PY'
import json, os, sys
create_gpu = os.environ['TF_CREATE_GPU_NODE'].lower() == 'true'
single = os.environ.get('TF_GPU_FLAVOR_ID', '').strip()
raw = os.environ.get('TF_GPU_FLAVOR_IDS_JSON', '{}').strip() or '{}'
try:
    per_node = json.loads(raw)
except json.JSONDecodeError as exc:
    print(f'TF_GPU_FLAVOR_IDS_JSON must be valid JSON map: {exc}', file=sys.stderr)
    sys.exit(1)
if not isinstance(per_node, dict):
    print('TF_GPU_FLAVOR_IDS_JSON must be a JSON object map.', file=sys.stderr)
    sys.exit(1)
if create_gpu and not single and not per_node:
    print('Set TF_GPU_FLAVOR_ID or TF_GPU_FLAVOR_IDS_JSON when TF_CREATE_GPU_NODE=true.', file=sys.stderr)
    sys.exit(1)
PY

echo "Prerequisites passed."


## Terraform Phase

Cells below run `terraform init`, `plan`, and `apply` in `devops/tf/kvm` using the parameter values from this notebook.

The current Terraform setup expects these networking behaviors:
- `allow-subnet-traffic-proj22` is attached to all node `sharednet1` ports.
- `allow-nat-proj22` is attached to `node1` `sharednet1` only.
- Private network port security is disabled on `private_net` and private ports.


In [ ]:
set -euo pipefail

cd "$TF_DIR"
terraform init

In [ ]:
set -euo pipefail
cd "$TF_DIR"

export TF_VAR_openstack_cloud="$TF_OPENSTACK_CLOUD"
export TF_VAR_openstack_region="$TF_OPENSTACK_REGION"
export TF_VAR_openstack_endpoint_type="$TF_OPENSTACK_ENDPOINT_TYPE"
export TF_VAR_suffix="$TF_SUFFIX"
export TF_VAR_cpu_flavor_id="$TF_CPU_FLAVOR_ID"
export TF_VAR_gpu_flavor_id="$TF_GPU_FLAVOR_ID"
export TF_VAR_gpu_flavor_ids="$TF_GPU_FLAVOR_IDS_JSON"
export TF_VAR_create_gpu_node="$TF_CREATE_GPU_NODE"

terraform plan


In [ ]:
set -euo pipefail
cd "$TF_DIR"

export TF_VAR_openstack_cloud="$TF_OPENSTACK_CLOUD"
export TF_VAR_openstack_region="$TF_OPENSTACK_REGION"
export TF_VAR_openstack_endpoint_type="$TF_OPENSTACK_ENDPOINT_TYPE"
export TF_VAR_suffix="$TF_SUFFIX"
export TF_VAR_cpu_flavor_id="$TF_CPU_FLAVOR_ID"
export TF_VAR_gpu_flavor_id="$TF_GPU_FLAVOR_ID"
export TF_VAR_gpu_flavor_ids="$TF_GPU_FLAVOR_IDS_JSON"
export TF_VAR_create_gpu_node="$TF_CREATE_GPU_NODE"

terraform apply


In [ ]:
set -euo pipefail

cd "$TF_DIR"
terraform output || true

## Validation + Recovery

Validation checklist:
- Terraform apply completed successfully.
- Terraform outputs include values needed to reach `node1`.
- `gpu_private_ips` contains both `gpu-node1` (`192.168.1.16`) and `gpu-node2` (`192.168.1.17`) when `TF_CREATE_GPU_NODE=true`.
- If using per-node GPU flavors, confirm Terraform plan shows each GPU node with its intended `flavor_id`.
- Terraform plan/apply reflect private-network `port_security_enabled = false` and expected `sharednet1` security group attachments.

Recovery note:
- If apply partially fails, use the cleanup guidance in `devops/tf/kvm/README.md` before retrying.


## Handoff to `node1`

Next steps:
1. SSH into `node1` (the control host/public entrypoint).
2. Clone this repository onto `node1`.
3. `cd` into repository root on `node1`.
4. Continue with Notebook 2: `02_node1_cluster_bootstrap.ipynb`.